# Day 1 — Titanic EDA + baseline

Goals:
- Load Titanic, do thorough EDA
- Establish a `DummyClassifier` baseline
- Train a simple LogisticRegression and beat the baseline
- Practice train/val split and the standard metrics


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
df = sns.load_dataset('titanic')
df.head()


## 1. Quick structural overview

In [ ]:
print('shape:', df.shape)
print('\ndtypes:'); print(df.dtypes)
print('\nmissing %:'); print((df.isna().mean() * 100).round(1).sort_values(ascending=False))
df.describe(include='all').T


## 2. Target distribution

In [ ]:
ax = sns.countplot(data=df, x='survived')
ax.set_title(f"Survived rate = {df['survived'].mean():.2%}")


## 3. Survival by feature

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flat, ['sex', 'pclass', 'embarked', 'sibsp', 'parch', 'alone']):
    sns.barplot(data=df, x=col, y='survived', ax=ax, errorbar=None)
    ax.set_title(f'survival rate by {col}')
plt.tight_layout()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.kdeplot(data=df, x='age', hue='survived', common_norm=False, ax=axes[0])
axes[0].set_title('age distribution by survival')
sns.kdeplot(data=df.assign(log_fare=np.log1p(df['fare'])), x='log_fare', hue='survived', common_norm=False, ax=axes[1])
axes[1].set_title('log(1+fare) distribution by survival')
plt.tight_layout()


## 4. Correlations (numeric only)

In [ ]:
num = df.select_dtypes(include='number')
sns.heatmap(num.corr(), annot=True, cmap='coolwarm', center=0); plt.title('numeric correlations');


## 5. Train/val split + dummy baseline

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

use = df.dropna(subset=['age', 'embarked']).copy()
y = use['survived']
X = pd.get_dummies(use[['pclass','sex','age','sibsp','parch','fare','embarked']], columns=['sex','embarked'], drop_first=True)

X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

dummy = DummyClassifier(strategy='most_frequent').fit(X_tr, y_tr)
p = dummy.predict(X_va)
print(f'baseline acc={accuracy_score(y_va, p):.3f}  f1={f1_score(y_va, p):.3f}')


## 6. Logistic regression (numeric features only)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

lr = Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))])
lr.fit(X_tr, y_tr)
p  = lr.predict(X_va)
pp = lr.predict_proba(X_va)[:, 1]
print(f'logreg  acc={accuracy_score(y_va, p):.3f}  f1={f1_score(y_va, p):.3f}  auc={roc_auc_score(y_va, pp):.3f}')
print(classification_report(y_va, p))


### Exercises
- Add `family_size = sibsp + parch + 1` and re-train; does AUC improve?
- Replace LogisticRegression with `RandomForestClassifier`. Compare metrics.
- Plot the ROC curve with `sklearn.metrics.RocCurveDisplay.from_estimator`.
- Push this notebook to your `ml-2-weeks` GitHub repo as `day01_titanic_baseline.ipynb`.
